In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/train.csv.zip
/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/sampleSubmission.csv.zip
/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/stores.csv
/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/features.csv.zip
/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/test.csv.zip


In [2]:
!pip install -q dagshub mlflow
import os
import time
from collections import deque, defaultdict
import numpy as np
import pandas as pd
import xgboost as xgb
import optuna

import dagshub
import mlflow
import mlflow.xgboost

optuna.logging.set_verbosity(optuna.logging.WARNING)
pd.set_option('display.max_columns', 50)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.3/273.3 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 77.8 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 75.1 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 46.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.3/121.3 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━

In [3]:
DATA_DIR = "/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting"
OUTPUT_DIR = "./outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

RANDOM_STATE = 42
# VAL_START = pd.Timestamp("2011-10-01")  
# VAL_END = pd.Timestamp("2012-01-15")
VAL_START = pd.Timestamp("2012-01-27")  
VAL_END = pd.Timestamp("2012-10-26")
N_HPO_TRIALS = 15
CHRISTMAS_SHIFT_FRACTION = 1 / 7

In [4]:
dagshub.init(repo_owner='lshek22', repo_name='walmart-recruiting-store-sales-forecasting', mlflow=True)
mlflow.set_experiment("XGBoost_v4_Training")

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=8f8e24c6-038e-4dda-973c-0a5cde92ee53&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=328b53f527f515e0ec4ace46cb07afe638e6148d85f9da11bb94c9e7e5953cc5




Accessing as lshek22

Initialized MLflow to track repo "lshek22/walmart-recruiting-store-sales-forecasting"

Repository lshek22/walmart-recruiting-store-sales-forecasting initialized!

<Experiment: artifact_location='mlflow-artifacts:/506fc1f9e2a64831814e837e05f2be71', creation_time=1784812407834, effective_trace_archival_retention=None, experiment_id='38', last_update_time=1784812407834, lifecycle_stage='active', name='XGBoost_v4_Training', tags={}, trace_location=None, workspace='default'>

In [5]:
train_raw = pd.read_csv(os.path.join(DATA_DIR, "train.csv.zip"), parse_dates=["Date"])
test_raw = pd.read_csv(os.path.join(DATA_DIR, "test.csv.zip"), parse_dates=["Date"])
stores = pd.read_csv(os.path.join(DATA_DIR, "stores.csv"))
features = pd.read_csv(os.path.join(DATA_DIR, "features.csv.zip"), parse_dates=["Date"])
features["IsHoliday"] = features["IsHoliday"].astype(bool)

print("train:", train_raw.shape, "test:", test_raw.shape, "stores:", stores.shape, "features:", features.shape)
train_raw.head()

train: (421570, 5) test: (115064, 4) stores: (45, 3) features: (8190, 12)


,Store,Dept,Date,Weekly_Sales,IsHoliday
0,1,1,2010-02-05,24924.50,False
1,1,1,2010-02-12,46039.49,True
2,1,1,2010-02-19,41595.55,False
3,1,1,2010-02-26,19403.54,False
4,1,1,2010-03-05,21827.90,False


In [6]:
def clean_features(features_df):
    df = features_df.copy()
    markdown_cols = [c for c in df.columns if c.startswith("MarkDown")]

    n_missing_before = int(df[markdown_cols].isna().sum().sum())
    n_negative = int((df[markdown_cols] < 0).sum().sum())
    df[markdown_cols] = df[markdown_cols].fillna(0).clip(lower=0)

    for col in ["CPI", "Unemployment"]:
        df[col] = df.groupby("Store")[col].transform(lambda s: s.ffill().bfill())

    stats = {
        "markdown_missing_before": n_missing_before,
        "markdown_negative_values_clipped": n_negative,
        "markdown_missing_after": int(df[markdown_cols].isna().sum().sum()),
        "cpi_unemployment_missing_after": int(df[["CPI", "Unemployment"]].isna().sum().sum()),
        "n_rows": len(df),
    }
    return df, stats


mlflow.set_experiment("XGBoost_v4_Cleaning")
with mlflow.start_run(run_name="clean_features"):
    features_clean, clean_stats = clean_features(features)
    mlflow.log_params({
        "markdown_fill_strategy": "fillna_0_clip_negative",
        "cpi_unemployment_fill_strategy": "groupby_store_ffill_bfill",
    })
    mlflow.log_metrics(clean_stats)

print(clean_stats)

🏃 View run clean_features at: https://dagshub.com/lshek22/walmart-recruiting-store-sales-forecasting.mlflow/#/experiments/39/runs/ffadcdc75dd049f0a71a2fa079e85a33
🧪 View experiment at: https://dagshub.com/lshek22/walmart-recruiting-store-sales-forecasting.mlflow/#/experiments/39
{'markdown_missing_before': 22870, 'markdown_negative_values_clipped': 44, 'markdown_missing_after': 0, 'cpi_unemployment_missing_after': 0, 'n_rows': 8190}


In [7]:
HOLIDAY_DATES = {
    "SuperBowl": ["2010-02-12", "2011-02-11", "2012-02-10", "2013-02-08"],
    "LaborDay": ["2010-09-10", "2011-09-09", "2012-09-07", "2013-09-06"],
    "Thanksgiving": ["2010-11-26", "2011-11-25", "2012-11-23", "2013-11-29"],
    "Christmas": ["2010-12-31", "2011-12-30", "2012-12-28", "2013-12-27"],
}
HOLIDAY_FLAG_COLS = [f"Is{name}" for name in HOLIDAY_DATES]
LAG_COLS = [
    "lag_1", "lag_2", "lag_4", "lag_8", "lag_52", 
    "rolling_mean_4", "rolling_mean_8", "rolling_std_4", 
    "storedept_avg_sales", "dept_woy_avg_sales"
]
CATEGORICAL_FEATURES = ["Store", "Dept", "Type"] + HOLIDAY_FLAG_COLS + ["IsHoliday"]
FEATURE_COLS = (
    ["Store", "Dept", "Type", "Size", "Temperature", "Fuel_Price",
     "MarkDown1", "MarkDown2", "MarkDown3", "MarkDown4", "MarkDown5",
     "CPI", "Unemployment", "IsHoliday", "Year", "Month", "WeekOfYear"]
    + HOLIDAY_FLAG_COLS + LAG_COLS
)


def add_static_features(df, stores_df, features_df):
    out = df.merge(stores_df, on="Store", how="left")
    out = out.merge(features_df.drop(columns=["IsHoliday"]), on=["Store", "Date"], how="left")
    for name, dates in HOLIDAY_DATES.items():
        out[f"Is{name}"] = out["Date"].isin(pd.to_datetime(dates))
    out["Year"] = out["Date"].dt.year
    out["Month"] = out["Date"].dt.month
    out["WeekOfYear"] = out["Date"].dt.isocalendar().week.astype(int)
    out["IsHoliday"] = out["IsHoliday"].astype(bool)
    return out


def add_bulk_lag_features(df):
    df = df.sort_values(["Store", "Dept", "Date"]).reset_index(drop=True)
    g = df.groupby(["Store", "Dept"])["Weekly_Sales"]
    df["lag_1"] = g.shift(1)
    df["lag_2"] = g.shift(2)
    df["lag_4"] = g.shift(4)
    df["lag_8"] = g.shift(8)
    df["lag_52"] = g.shift(52)
    df["rolling_mean_4"] = df.groupby(["Store", "Dept"])["Weekly_Sales"].transform(lambda s: s.shift(1).rolling(4, min_periods=1).mean())
    df["rolling_mean_8"] = df.groupby(["Store", "Dept"])["Weekly_Sales"].transform(lambda s: s.shift(1).rolling(8, min_periods=1).mean())
    df["rolling_std_4"] = df.groupby(["Store", "Dept"])["Weekly_Sales"].transform(lambda s: s.shift(1).rolling(4, min_periods=1).std())
    df["storedept_avg_sales"] = df.groupby(["Store", "Dept"])["Weekly_Sales"].transform(lambda s: s.expanding().mean().shift(1))

    tmp = df.sort_values(["Dept", "WeekOfYear", "Year"])
    tmp["dept_woy_avg_sales"] = tmp.groupby(["Dept", "WeekOfYear"])["Weekly_Sales"].transform(lambda s: s.expanding().mean().shift(1))
    df = tmp.sort_values(["Store", "Dept", "Date"]).reset_index(drop=True)
    return df


mlflow.set_experiment("XGBoost_v4_Feature_Engineering")
with mlflow.start_run(run_name="engineer_features"):
    train_static = add_static_features(train_raw, stores, features_clean)
    test_static = add_static_features(test_raw, stores, features_clean)

    train_m = add_bulk_lag_features(train_static)

    fillna_medians = {c: float(train_m[c].median()) for c in LAG_COLS}
    n_missing_before = int(train_m[LAG_COLS].isna().sum().sum())
    for c in LAG_COLS:
        train_m[c] = train_m.groupby(["Store", "Dept"])[c].transform(lambda s: s.fillna(s.median()))
        train_m[c] = train_m[c].fillna(fillna_medians[c])
    n_missing_after = int(train_m[LAG_COLS].isna().sum().sum())

    for c in CATEGORICAL_FEATURES:
        if train_m[c].dtype == "bool":
            train_m[c] = train_m[c].astype(str)
            test_static[c] = test_static[c].astype(str)

        train_m[c] = train_m[c].astype("category")
        test_static[c] = test_static[c].astype("category")

    mlflow.log_params({
        "lag_features": "lag_1,lag_2,lag_4,lag_8,lag_52",
        "rolling_features": "rolling_mean_4,rolling_mean_8,rolling_std_4",
        "target_encodings": "storedept_avg_sales, dept_woy_avg_sales (both expanding, shifted)",
        "test_lag_features": "computed recursively at forecast time",
    })
    mlflow.log_metrics({
        "n_train_rows": len(train_m),
        "n_test_rows": len(test_static),
        "n_lag_missing_before_fill": n_missing_before,
        "n_lag_missing_after_fill": n_missing_after,
    })

print(f"train_m: {train_m.shape}, test_static: {test_static.shape}")

🏃 View run engineer_features at: https://dagshub.com/lshek22/walmart-recruiting-store-sales-forecasting.mlflow/#/experiments/40/runs/6315e2000a5c4f25aa0dd3f3a4e012ab
🧪 View experiment at: https://dagshub.com/lshek22/walmart-recruiting-store-sales-forecasting.mlflow/#/experiments/40
train_m: (421570, 33), test_static: (115064, 22)


In [8]:
def select_features(train_df, candidate_features, random_state=42):
    X = train_df[candidate_features]
    y = train_df["Weekly_Sales"]
    w = np.where(train_df["IsHoliday"], 5, 1)

    probe_model = xgb.XGBRegressor(
        objective="reg:absoluteerror",
        n_estimators=150,
        enable_categorical=True,
        tree_method="hist",
        random_state=random_state
    )
    probe_model.fit(X, y, sample_weight=w)

    importances = pd.Series(probe_model.feature_importances_, index=candidate_features).sort_values(ascending=False)
    selected = importances[importances > 0].index.tolist()
    return selected, importances


mlflow.set_experiment("XGBoost_v4_Feature_Selection")
with mlflow.start_run(run_name="select_features"):
    selected_features, importances = select_features(train_m, FEATURE_COLS, RANDOM_STATE)

    mlflow.log_param("n_candidate_features", len(FEATURE_COLS))
    mlflow.log_param("n_selected_features", len(selected_features))
    mlflow.log_param("selected_features", ",".join(selected_features))
    mlflow.log_metrics({f"importance_{k}": float(v) for k, v in importances.items()})

    importance_path = os.path.join(OUTPUT_DIR, "feature_importances_xgb_v4.csv")
    importances.to_csv(importance_path, header=["importance"])
    mlflow.log_artifact(importance_path)

selected_cat_features = [c for c in CATEGORICAL_FEATURES if c in selected_features]
print("Selected features:", selected_features)

🏃 View run select_features at: https://dagshub.com/lshek22/walmart-recruiting-store-sales-forecasting.mlflow/#/experiments/41/runs/575bdf4c2500456ab0594afcffd04449
🧪 View experiment at: https://dagshub.com/lshek22/walmart-recruiting-store-sales-forecasting.mlflow/#/experiments/41
Selected features: ['lag_1', 'lag_2', 'lag_52', 'rolling_mean_4', 'rolling_mean_8', 'dept_woy_avg_sales', 'WeekOfYear', 'lag_4', 'IsHoliday', 'Dept', 'IsThanksgiving', 'Month', 'storedept_avg_sales', 'MarkDown3', 'Year', 'rolling_std_4', 'Store', 'MarkDown4', 'IsSuperBowl', 'Fuel_Price', 'MarkDown2', 'MarkDown1', 'lag_8', 'MarkDown5', 'Unemployment', 'CPI', 'Temperature', 'IsLaborDay']


In [9]:
class SeriesStore:
    def __init__(self):
        self.recent = defaultdict(lambda: deque(maxlen=8))   
        self.by_date = {}                                    
        self.sd_sum_count = defaultdict(lambda: [0.0, 0])    
        self.woy_sum_count = defaultdict(lambda: [0.0, 0]) 

    def get_features(self, store, dept, date, woy):
        key = (store, dept)
        recent = self.recent[key]
        n = len(recent)
        lag_1 = recent[-1] if n >= 1 else np.nan
        lag_2 = recent[-2] if n >= 2 else np.nan
        lag_4 = recent[-4] if n >= 4 else np.nan
        lag_8 = recent[-8] if n >= 8 else np.nan
        lag_52 = self.by_date.get((store, dept, date - pd.Timedelta(weeks=52)), np.nan)
        rolling_mean_4 = float(np.mean(list(recent)[-4:])) if n >= 1 else np.nan
        rolling_mean_8 = float(np.mean(list(recent))) if n >= 1 else np.nan
        rolling_std_4 = float(np.std(list(recent)[-4:])) if n >= 2 else np.nan
        s, c = self.sd_sum_count[key]
        storedept_avg = s / c if c > 0 else np.nan
        ws, wc = self.woy_sum_count[(dept, woy)]
        dept_woy_avg = ws / wc if wc > 0 else np.nan
        return dict(lag_1=lag_1, lag_2=lag_2, lag_4=lag_4, lag_8=lag_8, lag_52=lag_52,
                    rolling_mean_4=rolling_mean_4, rolling_mean_8=rolling_mean_8,
                    rolling_std_4=rolling_std_4, storedept_avg_sales=storedept_avg,
                    dept_woy_avg_sales=dept_woy_avg)

    def add(self, store, dept, date, woy, value):
        key = (store, dept)
        self.recent[key].append(value)
        self.by_date[(store, dept, date)] = value
        self.sd_sum_count[key][0] += value
        self.sd_sum_count[key][1] += 1
        self.woy_sum_count[(dept, woy)][0] += value
        self.woy_sum_count[(dept, woy)][1] += 1


def build_series_store(history_df):
    store = SeriesStore()
    for row in history_df.itertuples(index=False):
        store.add(row.Store, row.Dept, row.Date, row.WeekOfYear, row.Weekly_Sales)
    return store


def recursive_forecast(model, series_store, future_df, feature_cols, categorical_features, fillna_medians):
    preds_all = []
    for dt, day_rows in future_df.groupby("Date"):
        feat_dicts = [
            series_store.get_features(r.Store, r.Dept, dt, r.WeekOfYear)
            for r in day_rows.itertuples(index=False)
        ]
        feat_df = pd.DataFrame(feat_dicts, index=day_rows.index)
        merged = pd.concat([day_rows, feat_df], axis=1)

        for c in LAG_COLS:
            merged[c] = merged[c].fillna(fillna_medians.get(c, 0))
        for c in categorical_features:
            if merged[c].dtype == 'bool':
                merged[c] = merged[c].astype(str)
            merged[c] = merged[c].astype("category")

        preds = model.predict(merged[feature_cols])
        merged["Weekly_Sales_pred"] = preds
        preds_all.append(merged)

        for r, p in zip(day_rows.itertuples(index=False), preds):
            series_store.add(r.Store, r.Dept, dt, r.WeekOfYear, p)

    return pd.concat(preds_all, ignore_index=True)


def wmae(y_true, y_pred, is_holiday):
    weights = np.where(is_holiday, 5, 1)
    return float(np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights))

In [10]:
tr_for_val = train_m[train_m["Date"] < VAL_START].copy()
val_actual = train_m[(train_m["Date"] >= VAL_START) & (train_m["Date"] <= VAL_END)].copy()
val_future = val_actual.drop(columns=LAG_COLS + ["Weekly_Sales"])

w_tr_for_val = np.where(tr_for_val["IsHoliday"], 5, 1)
print(f"HPO train rows: {len(tr_for_val)}, walk-forward val rows: {len(val_actual)}")

mlflow.set_experiment("XGBoost_v4_HPO")
with mlflow.start_run(run_name="optuna_search"):

    def objective(trial):
        params = dict(
            objective="reg:absoluteerror",
            n_estimators=250,
            learning_rate=trial.suggest_float("learning_rate", 0.02, 0.15, log=True),
            max_depth=trial.suggest_int("max_depth", 4, 12),
            min_child_weight=trial.suggest_int("min_child_weight", 1, 30),
            subsample=trial.suggest_float("subsample", 0.6, 1.0),
            colsample_bytree=trial.suggest_float("colsample_bytree", 0.6, 1.0),
            enable_categorical=True,
            tree_method="hist",
            random_state=RANDOM_STATE,
        )
        model = xgb.XGBRegressor(**params)
        model.fit(tr_for_val[selected_features], tr_for_val["Weekly_Sales"], sample_weight=w_tr_for_val)

        series_store = build_series_store(tr_for_val[["Store", "Dept", "Date", "WeekOfYear", "Weekly_Sales"]])
        preds_df = recursive_forecast(model, series_store, val_future, selected_features, selected_cat_features, fillna_medians)
        merged = preds_df.merge(val_actual[["Store", "Dept", "Date", "Weekly_Sales"]], on=["Store", "Dept", "Date"])
        score = wmae(merged["Weekly_Sales"].values, merged["Weekly_Sales_pred"].values, merged["IsHoliday"].astype(bool).values)

        with mlflow.start_run(run_name=f"trial_{trial.number}", nested=True):
            mlflow.log_params(params)
            mlflow.log_metric("walkforward_val_wmae", score)
        return score

    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=N_HPO_TRIALS)

    mlflow.log_params({f"best_{k}": v for k, v in study.best_params.items()})
    mlflow.log_metric("best_walkforward_val_wmae", study.best_value)

best_params = dict(
    objective="reg:absoluteerror", 
    n_estimators=400, 
    enable_categorical=True, 
    tree_method="hist", 
    random_state=RANDOM_STATE, 
    **study.best_params
)
print("Best params:", study.best_params)
print("Best walk-forward validation WMAE:", study.best_value)

HPO train rows: 303030, walk-forward val rows: 118540
🏃 View run trial_0 at: https://dagshub.com/lshek22/walmart-recruiting-store-sales-forecasting.mlflow/#/experiments/42/runs/a30048e778f1409cb84f145664b37ed6
🧪 View experiment at: https://dagshub.com/lshek22/walmart-recruiting-store-sales-forecasting.mlflow/#/experiments/42
🏃 View run trial_1 at: https://dagshub.com/lshek22/walmart-recruiting-store-sales-forecasting.mlflow/#/experiments/42/runs/6ffbaef8481c47cc98107452038f3e59
🧪 View experiment at: https://dagshub.com/lshek22/walmart-recruiting-store-sales-forecasting.mlflow/#/experiments/42
🏃 View run trial_2 at: https://dagshub.com/lshek22/walmart-recruiting-store-sales-forecasting.mlflow/#/experiments/42/runs/307e7d7d73444c22ba602d42a8cc52b8
🧪 View experiment at: https://dagshub.com/lshek22/walmart-recruiting-store-sales-forecasting.mlflow/#/experiments/42
🏃 View run trial_3 at: https://dagshub.com/lshek22/walmart-recruiting-store-sales-forecasting.mlflow/#/experiments/42/runs/86f8

In [11]:
mlflow.set_experiment("XGBoost_v4_Training")
with mlflow.start_run(run_name="train_and_blend"):
    mlflow.log_params(best_params)
    mlflow.log_param("val_start", str(VAL_START.date()))
    mlflow.log_param("val_end", str(VAL_END.date()))
    mlflow.log_param("christmas_shift_fraction", CHRISTMAS_SHIFT_FRACTION)

    val_model = xgb.XGBRegressor(**best_params)
    val_model.fit(tr_for_val[selected_features], tr_for_val["Weekly_Sales"], sample_weight=w_tr_for_val)

    series_store_val = build_series_store(tr_for_val[["Store", "Dept", "Date", "WeekOfYear", "Weekly_Sales"]])
    val_preds_df = recursive_forecast(val_model, series_store_val, val_future, selected_features, selected_cat_features, fillna_medians)
    val_merged = val_preds_df.merge(val_actual[["Store", "Dept", "Date", "Weekly_Sales"]], on=["Store", "Dept", "Date"])
    val_wmae = wmae(val_merged["Weekly_Sales"].values, val_merged["Weekly_Sales_pred"].values, val_merged["IsHoliday"].astype(bool).values)
    mlflow.log_metric("val_wmae_model_only", val_wmae)

    best_alpha, best_blend_wmae = 1.0, val_wmae
    if "lag_52" in val_merged.columns:
        naive_val = val_merged["lag_52"].values
        for alpha in np.linspace(0, 1, 21):
            blend = alpha * val_merged["Weekly_Sales_pred"].values + (1 - alpha) * naive_val
            score = wmae(val_merged["Weekly_Sales"].values, blend, val_merged["IsHoliday"].astype(bool).values)
            if score < best_blend_wmae:
                best_alpha, best_blend_wmae = alpha, score
    mlflow.log_metric("best_blend_alpha", best_alpha)
    mlflow.log_metric("val_wmae_blended", best_blend_wmae)

    w_full = np.where(train_m["IsHoliday"], 5, 1)
    final_model = xgb.XGBRegressor(**best_params)
    final_model.fit(train_m[selected_features], train_m["Weekly_Sales"], sample_weight=w_full)
    mlflow.xgboost.log_model(final_model, artifact_path="model")

    t0 = time.time()
    series_store_test = build_series_store(train_m[["Store", "Dept", "Date", "WeekOfYear", "Weekly_Sales"]])
    test_preds_df = recursive_forecast(final_model, series_store_test, test_static, selected_features, selected_cat_features, fillna_medians)
    mlflow.log_metric("test_forecast_seconds", time.time() - t0)

    if "lag_52" in test_preds_df.columns:
        test_blend = best_alpha * test_preds_df["Weekly_Sales_pred"].values + (1 - best_alpha) * test_preds_df["lag_52"].values
    else:
        test_blend = test_preds_df["Weekly_Sales_pred"].values

    def christmas_shift_correction(df, preds, shift_fraction=CHRISTMAS_SHIFT_FRACTION):
        preds = preds.copy()
        df = df.reset_index(drop=True)
        idx_christmas = np.where(df["IsChristmas"].astype(bool).values)[0]
        for i in idx_christmas:
            store, dept, date = df.loc[i, "Store"], df.loc[i, "Dept"], df.loc[i, "Date"]
            prev_mask = (df["Store"] == store) & (df["Dept"] == dept) & (df["Date"] == date - pd.Timedelta(weeks=1))
            prev_idx = df.index[prev_mask]
            if len(prev_idx) == 1:
                shift_amt = preds[i] * shift_fraction
                preds[i] -= shift_amt
                preds[prev_idx[0]] += shift_amt
        return preds

    test_preds_reset = test_preds_df.reset_index(drop=True)
    test_final_preds = christmas_shift_correction(test_preds_reset, test_blend, CHRISTMAS_SHIFT_FRACTION)

    submission = test_preds_reset.copy()
    submission["Weekly_Sales"] = test_final_preds
    submission["Id"] = (
        submission["Store"].astype(int).astype(str) + "_" +
        submission["Dept"].astype(int).astype(str) + "_" +
        submission["Date"].dt.strftime("%Y-%m-%d")
    )
    submission = submission[["Id", "Weekly_Sales"]]

    submission_path = os.path.join(OUTPUT_DIR, "submission_xgb_v4.csv")
    submission.to_csv(submission_path, index=False)
    mlflow.log_artifact(submission_path)

print(f"Model-only walk-forward validation WMAE: {val_wmae:.2f}")
print(f"Blended walk-forward validation WMAE:    {best_blend_wmae:.2f} (alpha={best_alpha:.2f})")
print(f"Saved {len(submission)} predictions to {submission_path}")

2026/07/25 16:48:04 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run train_and_blend at: https://dagshub.com/lshek22/walmart-recruiting-store-sales-forecasting.mlflow/#/experiments/38/runs/fce07372a2ed45c097f0547236c1e642
🧪 View experiment at: https://dagshub.com/lshek22/walmart-recruiting-store-sales-forecasting.mlflow/#/experiments/38
Model-only walk-forward validation WMAE: 1715.70
Blended walk-forward validation WMAE:    1687.97 (alpha=0.75)
Saved 115064 predictions to ./outputs/submission_xgb_v4.csv


In [12]:
submission.head()

,Id,Weekly_Sales
0,1_1_2012-11-02,40772.973809
1,1_2_2012-11-02,52191.198013
2,1_3_2012-11-02,10944.787666
3,1_4_2012-11-02,45553.061998
4,1_5_2012-11-02,35028.692988
